## XGBoost

In [3]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV   

train = pd.read_csv('../data/bot_identification/train.csv', sep=';')
test = pd.read_csv('../data/bot_identification/test.csv', sep=';')
# val = pd.read_csv('../data/bot_identification/val.csv', sep=';')

encoder = OneHotEncoder()
x_train = train['text']
x_test = test['text']
# x_val = val['text']

y_train = pd.DataFrame(train['account.type'])
y_test = pd.DataFrame(test['account.type'])
# y_val = pd.DataFrame(val['account.type'])
y_train_matrix = encoder.fit_transform(y_train).toarray().tolist()
y_test_matrix = encoder.transform(y_test).toarray().tolist()
# y_val_matrix = encoder.transform(y_val).toarray().tolist()

def get_single_label(encoded):
    y_labels = []
    for isBot, isntBot in encoded:
        y_labels.append(int(isBot))
    return y_labels
    
y_train_encoded = get_single_label(y_train_matrix)
y_test_encoded = get_single_label(y_test_matrix)
# y_val_encoded = get_single_label(y_val_matrix)

In [ ]:
# =========================================================
# Safe TF-IDF + XGBoost RandomizedSearchCV for 40-core node
# =========================================================

import os

# Put these BEFORE importing numpy / sklearn / xgboost when possible.
# If using Jupyter, restart the kernel before running this cell.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import json
import gc
import numpy as np
import joblib

from threadpoolctl import threadpool_limits

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, make_scorer

from xgboost import XGBClassifier


# =========================================================
# Helper for JSON serialization
# =========================================================

def make_json_serializable(obj):
    """Convert NumPy/scikit-learn objects into JSON-safe Python objects."""
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, tuple):
        return list(obj)
    return str(obj)


# =========================================================
# Pipeline
# =========================================================

pipeline = Pipeline([
    (
        "features",
        FeatureUnion(
            transformer_list=[
                (
                    "word_tfidf",
                    TfidfVectorizer(
                        analyzer="word",
                        lowercase=True,
                        dtype=np.float32
                    )
                ),
                (
                    "char_tfidf",
                    TfidfVectorizer(
                        analyzer="char_wb",
                        lowercase=True,
                        dtype=np.float32
                    )
                )
            ],

            # Keep this at 1 because RandomizedSearchCV is parallelized.
            n_jobs=1
        )
    ),
    (
        "xgb",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,

            # Safer 40-core strategy:
            # 5 CV jobs × 8 XGBoost threads ≈ 40 threads.
            n_jobs=8
        )
    )
])


# =========================================================
# Safer reduced parameter search space
# =========================================================

param_dist = {
    # -----------------------------
    # Word TF-IDF
    # -----------------------------
    "features__word_tfidf__ngram_range": [
        (1, 1),
        (1, 2)
    ],
    "features__word_tfidf__max_features": [
        20000,
        50000
    ],
    "features__word_tfidf__min_df": [
        2,
        3,
        5
    ],
    "features__word_tfidf__max_df": [
        0.90,
        0.95,
        1.0
    ],
    "features__word_tfidf__sublinear_tf": [
        True
    ],

    # -----------------------------
    # Character TF-IDF
    # -----------------------------
    "features__char_tfidf__ngram_range": [
        (3, 5),
        (4, 6)
    ],
    "features__char_tfidf__max_features": [
        20000,
        50000
    ],
    "features__char_tfidf__min_df": [
        2,
        3,
        5
    ],
    "features__char_tfidf__max_df": [
        0.90,
        0.95,
        1.0
    ],
    "features__char_tfidf__sublinear_tf": [
        True
    ],

    # -----------------------------
    # Word vs character weighting
    # -----------------------------
    "features__transformer_weights": [
        {"word_tfidf": 1.0, "char_tfidf": 1.0},
        {"word_tfidf": 1.0, "char_tfidf": 0.5},
        {"word_tfidf": 0.75, "char_tfidf": 1.0},
        {"word_tfidf": 1.25, "char_tfidf": 1.0},
        {"word_tfidf": 1.0, "char_tfidf": 1.25}
    ],

    # -----------------------------
    # XGBoost
    # -----------------------------
    "xgb__n_estimators": [
        300,
        500,
        800
    ],
    "xgb__max_depth": [
        3,
        4,
        5,
        6,
        8
    ],
    "xgb__learning_rate": [
        0.01,
        0.03,
        0.035,
        0.05,
        0.1
    ],
    "xgb__subsample": [
        0.7,
        0.8,
        0.85,
        0.9
    ],
    "xgb__colsample_bytree": [
        0.55,
        0.6,
        0.7,
        0.8
    ],
    "xgb__gamma": [
        0,
        0.1,
        0.3,
        0.5
    ],
    "xgb__min_child_weight": [
        1,
        2,
        3,
        5
    ],
    "xgb__reg_alpha": [
        0,
        0.01,
        0.1,
        0.3,
        1
    ],
    "xgb__reg_lambda": [
        1,
        5,
        10,
        20
    ],
    "xgb__scale_pos_weight": [
        1,
        2,
        2.5,
        5
    ],

    # Helps control memory for hist tree method
    "xgb__max_bin": [
        64,
        128,
        256
    ]
}


# =========================================================
# Cross-validation and scoring
# =========================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f1_scorer = make_scorer(
    f1_score,
    pos_label=1
)


# =========================================================
# Randomized search
# =========================================================

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,

    # 50 iterations × 5 folds = 250 total fits
    # Safer than 100 iterations while debugging cluster memory.
    n_iter=50,

    scoring=f1_scorer,
    cv=cv,

    # Safer 40-core strategy:
    # 5 parallel CV jobs × 8 XGBoost threads = about 40 total threads.
    # This avoids launching 40 huge TF-IDF/XGBoost jobs at once.
    n_jobs=5,
    pre_dispatch=5,

    verbose=3,
    random_state=42,
    refit=True,
    return_train_score=False,

    # Use "raise" while debugging. Change to np.nan for unattended long runs.
    error_score="raise"
)


# =========================================================
# Fit search
# =========================================================

print("Starting RandomizedSearchCV...")

# Limit BLAS libraries to 1 thread.
# XGBoost still uses its own n_jobs=8 setting.
with threadpool_limits(limits=1, user_api="blas"):
    search.fit(x_train, y_train_encoded)

print("Search complete.")






Starting RandomizedSearchCV...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 2/5] END features__char_tfidf__max_df=1.0, features__char_tfidf__max_features=50000, features__char_tfidf__min_df=5, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 1.0, 'char_tfidf': 1.0}, features__word_tfidf__max_df=1.0, features__word_tfidf__max_features=50000, features__word_tfidf__min_df=5, features__word_tfidf__ngram_range=(1, 1), features__word_tfidf__sublinear_tf=True, xgb__colsample_bytree=0.8, xgb__gamma=0.1, xgb__learning_rate=0.03, xgb__max_bin=256, xgb__max_depth=8, xgb__min_child_weight=3, xgb__n_estimators=500, xgb__reg_alpha=0.3, xgb__reg_lambda=20, xgb__scale_pos_weight=1, xgb__subsample=0.9;, score=0.816 total time= 6.9min
[CV 3/5] END features__char_tfidf__max_df=0.95, features__char_tfidf__max_features=50000, features__char_tfidf__min_df=3, features__char_tfidf__ngram_range=(4, 6), features_

NameError: name 'X_test' is not defined

[CV 4/5] END features__char_tfidf__max_df=1.0, features__char_tfidf__max_features=50000, features__char_tfidf__min_df=5, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 0.75, 'char_tfidf': 1.0}, features__word_tfidf__max_df=0.9, features__word_tfidf__max_features=20000, features__word_tfidf__min_df=5, features__word_tfidf__ngram_range=(1, 2), features__word_tfidf__sublinear_tf=True, xgb__colsample_bytree=0.6, xgb__gamma=0.5, xgb__learning_rate=0.035, xgb__max_bin=256, xgb__max_depth=3, xgb__min_child_weight=5, xgb__n_estimators=800, xgb__reg_alpha=0.3, xgb__reg_lambda=5, xgb__scale_pos_weight=1, xgb__subsample=0.8;, score=0.821 total time= 1.9min
[CV 4/5] END features__char_tfidf__max_df=0.9, features__char_tfidf__max_features=50000, features__char_tfidf__min_df=2, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 0.75, 'char_tfid

In [ ]:

# =========================================================
# Save best model
# =========================================================

best_model = search.best_estimator_

model_filename = "big_grid_model_40core_safe.joblib"
joblib.dump(best_model, model_filename, compress=3)

print(f"Saved best model to {model_filename}")


# =========================================================
# Evaluate on test set
# =========================================================

y_pred = best_model.predict(x_test)

report_dict = classification_report(
    y_test_encoded,
    y_pred,
    output_dict=True
)

report_text = classification_report(
    y_test_encoded,
    y_pred
)

results = {
    "best_cv_f1_score": search.best_score_,
    "best_parameters": search.best_params_,
    "classification_report": report_dict
}

# =========================================================
# Print summary
# =========================================================

print("\nBest CV F1 score:")
print(search.best_score_)

print("\nBest parameters:")
print(search.best_params_)

print("\nTest set classification report:")
print(report_text)

# =========================================================
# Save results as JSON
# =========================================================

results_filename = "xgb_search_results.json"

with open(results_filename, "w") as f:
    json.dump(
        results,
        f,
        indent=4,
        default=make_json_serializable
    )

print(f"Saved results to {results_filename}")


# =========================================================
# Clean up memory
# =========================================================

gc.collect()

Saved results to xgb_search_results.json

Best CV F1 score:
0.8552033566706131

Best parameters:
{'xgb__subsample': 0.9, 'xgb__scale_pos_weight': 2.5, 'xgb__reg_lambda': 10, 'xgb__reg_alpha': 0.1, 'xgb__n_estimators': 500, 'xgb__min_child_weight': 3, 'xgb__max_depth': 6, 'xgb__max_bin': 64, 'xgb__learning_rate': 0.1, 'xgb__gamma': 0.1, 'xgb__colsample_bytree': 0.7, 'features__word_tfidf__sublinear_tf': True, 'features__word_tfidf__ngram_range': (1, 1), 'features__word_tfidf__min_df': 2, 'features__word_tfidf__max_features': 20000, 'features__word_tfidf__max_df': 1.0, 'features__transformer_weights': {'word_tfidf': 1.0, 'char_tfidf': 0.5}, 'features__char_tfidf__sublinear_tf': True, 'features__char_tfidf__ngram_range': (3, 5), 'features__char_tfidf__min_df': 2, 'features__char_tfidf__max_features': 20000, 'features__char_tfidf__max_df': 0.95}

Test set classification report:
              precision    recall  f1-score   support

           0       0.95      0.72      0.82      1278
    

9

## saving xgb classifier and tfidf vectorizer used

In [ ]:
import joblib
from joblib import dump
 
filename = f"../models/bot_id_xgb_tree{f1}.joblib"

joblib.dump(pipeline, filename)

## linear svm

In [6]:
# =========================================================
# Safe TF-IDF + Linear SVM RandomizedSearchCV for 40-core node
# =========================================================

import os

# Put these BEFORE importing numpy / sklearn when possible.
# If using Jupyter, restart the kernel before running this cell.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import json
import gc
import numpy as np
import joblib

from threadpoolctl import threadpool_limits

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, make_scorer
from sklearn.svm import LinearSVC


# =========================================================
# Helper for JSON serialization
# =========================================================

def make_json_serializable(obj):
    """Convert NumPy/scikit-learn objects into JSON-safe Python objects."""
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, tuple):
        return list(obj)
    return str(obj)


# =========================================================
# Pipeline
# =========================================================

pipeline = Pipeline([
    (
        "features",
        FeatureUnion(
            transformer_list=[
                (
                    "word_tfidf",
                    TfidfVectorizer(
                        analyzer="word",
                        lowercase=True,
                        dtype=np.float32
                    )
                ),
                (
                    "char_tfidf",
                    TfidfVectorizer(
                        analyzer="char_wb",
                        lowercase=True,
                        dtype=np.float32
                    )
                )
            ],

            # Keep this at 1 because RandomizedSearchCV is parallelized.
            n_jobs=1
        )
    ),
    (
        "svm",
        LinearSVC(
            random_state=42,
            max_iter=10000
        )
    )
])


# =========================================================
# Valid LinearSVC parameter search spaces
# =========================================================
# Important:
# Not all LinearSVC penalty/loss/dual combinations are valid.
# So we use a list of dictionaries to avoid invalid combinations.

param_dist = [
    # -----------------------------------------------------
    # L2 penalty + squared hinge loss
    # Most common/default LinearSVC setup.
    # -----------------------------------------------------
    {
        # Word TF-IDF
        "features__word_tfidf__ngram_range": [
            (1, 1),
            (1, 2)
        ],
        "features__word_tfidf__max_features": [
            20000,
            50000,
            100000
        ],
        "features__word_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__word_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__word_tfidf__sublinear_tf": [
            True
        ],

        # Character TF-IDF
        "features__char_tfidf__ngram_range": [
            (3, 5),
            (4, 6)
        ],
        "features__char_tfidf__max_features": [
            20000,
            50000,
            100000
        ],
        "features__char_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__char_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__char_tfidf__sublinear_tf": [
            True
        ],

        # Word vs character weighting
        "features__transformer_weights": [
            {"word_tfidf": 1.0, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 0.5},
            {"word_tfidf": 0.75, "char_tfidf": 1.0},
            {"word_tfidf": 1.25, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 1.25}
        ],

        # Linear SVM
        "svm__penalty": ["l2"],
        "svm__loss": ["squared_hinge"],
        "svm__dual": [True, False],
        "svm__C": [
            0.001,
            0.003,
            0.01,
            0.03,
            0.1,
            0.3,
            1,
            3,
            10
        ],
        "svm__class_weight": [
            None,
            "balanced"
        ],
        "svm__tol": [
            1e-4,
            1e-3
        ],
        "svm__max_iter": [
            5000,
            10000,
            20000
        ]
    },

    # -----------------------------------------------------
    # L2 penalty + hinge loss
    # Classic linear SVM hinge loss.
    # Requires dual=True.
    # -----------------------------------------------------
    {
        # Word TF-IDF
        "features__word_tfidf__ngram_range": [
            (1, 1),
            (1, 2)
        ],
        "features__word_tfidf__max_features": [
            20000,
            50000,
            100000
        ],
        "features__word_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__word_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__word_tfidf__sublinear_tf": [
            True
        ],

        # Character TF-IDF
        "features__char_tfidf__ngram_range": [
            (3, 5),
            (4, 6)
        ],
        "features__char_tfidf__max_features": [
            20000,
            50000,
            100000
        ],
        "features__char_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__char_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__char_tfidf__sublinear_tf": [
            True
        ],

        # Word vs character weighting
        "features__transformer_weights": [
            {"word_tfidf": 1.0, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 0.5},
            {"word_tfidf": 0.75, "char_tfidf": 1.0},
            {"word_tfidf": 1.25, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 1.25}
        ],

        # Linear SVM
        "svm__penalty": ["l2"],
        "svm__loss": ["hinge"],
        "svm__dual": [True],
        "svm__C": [
            0.001,
            0.003,
            0.01,
            0.03,
            0.1,
            0.3,
            1,
            3,
            10
        ],
        "svm__class_weight": [
            None,
            "balanced"
        ],
        "svm__tol": [
            1e-4,
            1e-3
        ],
        "svm__max_iter": [
            5000,
            10000,
            20000
        ]
    },

    # -----------------------------------------------------
    # L1 penalty + squared hinge loss
    # Can perform feature selection by pushing some weights to zero.
    # Requires dual=False.
    # -----------------------------------------------------
    {
        # Word TF-IDF
        "features__word_tfidf__ngram_range": [
            (1, 1),
            (1, 2)
        ],
        "features__word_tfidf__max_features": [
            20000,
            50000
        ],
        "features__word_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__word_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__word_tfidf__sublinear_tf": [
            True
        ],

        # Character TF-IDF
        "features__char_tfidf__ngram_range": [
            (3, 5),
            (4, 6)
        ],
        "features__char_tfidf__max_features": [
            20000,
            50000
        ],
        "features__char_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__char_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__char_tfidf__sublinear_tf": [
            True
        ],

        # Word vs character weighting
        "features__transformer_weights": [
            {"word_tfidf": 1.0, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 0.5},
            {"word_tfidf": 0.75, "char_tfidf": 1.0},
            {"word_tfidf": 1.25, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 1.25}
        ],

        # Linear SVM
        "svm__penalty": ["l1"],
        "svm__loss": ["squared_hinge"],
        "svm__dual": [False],
        "svm__C": [
            0.001,
            0.003,
            0.01,
            0.03,
            0.1,
            0.3,
            1,
            3
        ],
        "svm__class_weight": [
            None,
            "balanced"
        ],
        "svm__tol": [
            1e-4,
            1e-3
        ],
        "svm__max_iter": [
            5000,
            10000,
            20000
        ]
    }
]


# =========================================================
# Cross-validation and scoring
# =========================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f1_scorer = make_scorer(
    f1_score,
    pos_label=1
)


# =========================================================
# Randomized search
# =========================================================

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,

    # 50 iterations × 5 folds = 250 total fits.
    # LinearSVC is usually much faster than XGBoost for TF-IDF,
    # but TF-IDF fitting can still use a lot of memory.
    n_iter=50,

    scoring=f1_scorer,
    cv=cv,

    # LinearSVC itself does not have n_jobs.
    # So parallelism mainly happens here.
    # Start with 20 for safety; increase to 30 or 40 if memory is fine.
    n_jobs=20,
    pre_dispatch=20,

    verbose=3,
    random_state=42,
    refit=True,
    return_train_score=False,

    # Use "raise" while debugging. Change to np.nan for unattended long runs.
    error_score="raise"
)


# =========================================================
# Fit search
# =========================================================

print("Starting LinearSVC RandomizedSearchCV...")

# Limit BLAS libraries to 1 thread.
with threadpool_limits(limits=1, user_api="blas"):
    search.fit(x_train, y_train_encoded)

print("Search complete.")


# =========================================================
# Save best model
# =========================================================

best_model = search.best_estimator_

model_filename = "linear_svm_tfidf_model_40core_safe.joblib"
joblib.dump(best_model, model_filename, compress=3)

print(f"Saved best model to {model_filename}")


# =========================================================
# Evaluate on test set
# =========================================================

y_pred = best_model.predict(x_test)

report_dict = classification_report(
    y_test_encoded,
    y_pred,
    output_dict=True
)

report_text = classification_report(
    y_test_encoded,
    y_pred
)

results = {
    "model_type": "LinearSVC",
    "best_cv_f1_score": search.best_score_,
    "best_parameters": search.best_params_,
    "classification_report": report_dict
}


# =========================================================
# Optional: get decision scores
# =========================================================
# These are NOT probabilities.
# For binary LinearSVC:
#   positive score = more confident class 1
#   negative score = more confident class 0

if hasattr(best_model, "decision_function"):
    decision_scores = best_model.decision_function(x_test)

    results["decision_scores_note"] = (
        "These are LinearSVC margin scores, not probabilities."
    )

    # Save only summary stats to avoid huge JSON files.
    results["decision_scores_summary"] = {
        "min": float(np.min(decision_scores)),
        "max": float(np.max(decision_scores)),
        "mean": float(np.mean(decision_scores)),
        "std": float(np.std(decision_scores))
    }


# =========================================================
# Save results as JSON
# =========================================================

results_filename = "linear_svm_search_results.json"

with open(results_filename, "w") as f:
    json.dump(
        results,
        f,
        indent=4,
        default=make_json_serializable
    )

print(f"Saved results to {results_filename}")


# =========================================================
# Print summary
# =========================================================

print("\nBest CV F1 score:")
print(search.best_score_)

print("\nBest parameters:")
print(search.best_params_)

print("\nTest set classification report:")
print(report_text)


# =========================================================
# Clean up memory
# =========================================================

gc.collect()

Starting LinearSVC RandomizedSearchCV...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 5/5] END features__char_tfidf__max_df=1.0, features__char_tfidf__max_features=20000, features__char_tfidf__min_df=3, features__char_tfidf__ngram_range=(3, 5), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 1.0, 'char_tfidf': 0.5}, features__word_tfidf__max_df=0.95, features__word_tfidf__max_features=100000, features__word_tfidf__min_df=2, features__word_tfidf__ngram_range=(1, 1), features__word_tfidf__sublinear_tf=True, svm__C=3, svm__class_weight=balanced, svm__dual=True, svm__loss=squared_hinge, svm__max_iter=20000, svm__penalty=l2, svm__tol=0.001;, score=0.816 total time=   6.2s
[CV 4/5] END features__char_tfidf__max_df=0.95, features__char_tfidf__max_features=20000, features__char_tfidf__min_df=3, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 1.0, 'char_tfid

146

## logistic regression

In [6]:
# =========================================================
# Safe TF-IDF + Logistic Regression RandomizedSearchCV
# For binary bot identification
# =========================================================

import os

# Put these BEFORE importing numpy / sklearn when possible.
# If using Jupyter, restart the kernel before running this cell.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import json
import gc
import numpy as np
import pandas as pd
import joblib

from scipy.stats import loguniform
from threadpoolctl import threadpool_limits

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    f1_score,
    make_scorer,
    confusion_matrix,
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression


# =========================================================
# Helper for JSON serialization
# =========================================================

def make_json_serializable(obj):
    """Convert NumPy/scikit-learn objects into JSON-safe Python objects."""
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, tuple):
        return list(obj)
    return str(obj)


# =========================================================
# Pipeline
# =========================================================

pipeline = Pipeline([
    (
        "features",
        FeatureUnion(
            transformer_list=[
                (
                    "word_tfidf",
                    TfidfVectorizer(
                        analyzer="word",
                        lowercase=True,
                        dtype=np.float32
                    )
                ),
                (
                    "char_tfidf",
                    TfidfVectorizer(
                        analyzer="char_wb",
                        lowercase=True,
                        dtype=np.float32
                    )
                )
            ],

            # Keep this at 1 because RandomizedSearchCV is parallelized.
            n_jobs=1
        )
    ),
    (
        "logreg",
        LogisticRegression(
            solver="saga",
            random_state=42,
            n_jobs=1,
            max_iter=3000,
            verbose=0
        )
    )
])


# =========================================================
# Parameter search space
# =========================================================
# RandomizedSearchCV supports a list of dictionaries for param_distributions.
# This lets us avoid invalid combinations like l1_ratio with penalty="l2".
# Scikit-learn samples one dictionary, then samples parameters from that dictionary.

param_dist = [
    # =====================================================
    # L2 Logistic Regression
    # =====================================================
    {
        # -----------------------------
        # Word TF-IDF
        # -----------------------------
        "features__word_tfidf__ngram_range": [
            (1, 1),
            (1, 2)
        ],
        "features__word_tfidf__max_features": [
            20000,
            50000,
            80000
        ],
        "features__word_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__word_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__word_tfidf__sublinear_tf": [
            True
        ],

        # -----------------------------
        # Character TF-IDF
        # -----------------------------
        "features__char_tfidf__ngram_range": [
            (3, 5),
            (4, 6)
        ],
        "features__char_tfidf__max_features": [
            20000,
            50000,
            80000
        ],
        "features__char_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__char_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__char_tfidf__sublinear_tf": [
            True
        ],

        # -----------------------------
        # Word vs character weighting
        # -----------------------------
        "features__transformer_weights": [
            {"word_tfidf": 1.0, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 0.5},
            {"word_tfidf": 0.75, "char_tfidf": 1.0},
            {"word_tfidf": 1.25, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 1.25}
        ],

        # -----------------------------
        # Logistic Regression
        # -----------------------------
        "logreg__penalty": [
            "l2"
        ],
        "logreg__C": loguniform(1e-3, 100),
        "logreg__class_weight": [
            None,
            "balanced"
        ],
        "logreg__tol": [
            1e-3,
            1e-4
        ],
        "logreg__max_iter": [
            1000,
            2000,
            3000
        ]
    },

    # =====================================================
    # L1 Logistic Regression
    # =====================================================
    {
        "features__word_tfidf__ngram_range": [
            (1, 1),
            (1, 2)
        ],
        "features__word_tfidf__max_features": [
            20000,
            50000,
            80000
        ],
        "features__word_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__word_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__word_tfidf__sublinear_tf": [
            True
        ],

        "features__char_tfidf__ngram_range": [
            (3, 5),
            (4, 6)
        ],
        "features__char_tfidf__max_features": [
            20000,
            50000,
            80000
        ],
        "features__char_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__char_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__char_tfidf__sublinear_tf": [
            True
        ],

        "features__transformer_weights": [
            {"word_tfidf": 1.0, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 0.5},
            {"word_tfidf": 0.75, "char_tfidf": 1.0},
            {"word_tfidf": 1.25, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 1.25}
        ],

        "logreg__penalty": [
            "l1"
        ],
        "logreg__C": loguniform(1e-3, 100),
        "logreg__class_weight": [
            None,
            "balanced"
        ],
        "logreg__tol": [
            1e-3,
            1e-4
        ],
        "logreg__max_iter": [
            1000,
            2000,
            3000
        ]
    },

    # =====================================================
    # Elastic Net Logistic Regression
    # =====================================================
    {
        "features__word_tfidf__ngram_range": [
            (1, 1),
            (1, 2)
        ],
        "features__word_tfidf__max_features": [
            20000,
            50000,
            80000
        ],
        "features__word_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__word_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__word_tfidf__sublinear_tf": [
            True
        ],

        "features__char_tfidf__ngram_range": [
            (3, 5),
            (4, 6)
        ],
        "features__char_tfidf__max_features": [
            20000,
            50000,
            80000
        ],
        "features__char_tfidf__min_df": [
            2,
            3,
            5
        ],
        "features__char_tfidf__max_df": [
            0.90,
            0.95,
            1.0
        ],
        "features__char_tfidf__sublinear_tf": [
            True
        ],

        "features__transformer_weights": [
            {"word_tfidf": 1.0, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 0.5},
            {"word_tfidf": 0.75, "char_tfidf": 1.0},
            {"word_tfidf": 1.25, "char_tfidf": 1.0},
            {"word_tfidf": 1.0, "char_tfidf": 1.25}
        ],

        "logreg__penalty": [
            "elasticnet"
        ],
        "logreg__C": loguniform(1e-3, 100),
        "logreg__l1_ratio": [
            0.1,
            0.25,
            0.5,
            0.75,
            0.9
        ],
        "logreg__class_weight": [
            None,
            "balanced"
        ],
        "logreg__tol": [
            1e-3,
            1e-4
        ],
        "logreg__max_iter": [
            1000,
            2000,
            3000
        ]
    }
]


# =========================================================
# Cross-validation and scoring
# =========================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f1_scorer = make_scorer(
    f1_score,
    pos_label=1
)


# =========================================================
# Randomized search
# =========================================================

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,

    # 50 iterations × 5 folds = 250 total fits
    n_iter=50,

    scoring=f1_scorer,
    cv=cv,

    # Logistic Regression does not need nested XGBoost-style threading.
    # Increase to 20 if your RAM is fine.
    n_jobs=30,
    pre_dispatch=10,

    verbose=3,
    random_state=42,
    refit=True,
    return_train_score=False,

    # Use "raise" while debugging.
    # Change to np.nan for unattended long runs.
    error_score="raise"
)


# =========================================================
# Fit search
# =========================================================

print("Starting Logistic Regression RandomizedSearchCV...")

# Limit BLAS libraries to 1 thread to avoid hidden oversubscription.
with threadpool_limits(limits=1, user_api="blas"):
    search.fit(x_train, y_train_encoded)

print("Search complete.")


# =========================================================
# Best model
# =========================================================

best_model = search.best_estimator_


# =========================================================
# Evaluate on test set
# =========================================================

y_pred = best_model.predict(x_test)

report_dict = classification_report(
    y_test_encoded,
    y_pred,
    output_dict=True,
    zero_division=0
)

report_text = classification_report(
    y_test_encoded,
    y_pred,
    zero_division=0
)

conf_matrix = confusion_matrix(
    y_test_encoded,
    y_pred
)

# Logistic Regression supports predict_proba.
y_proba = best_model.predict_proba(x_test)

# Probability of class 1, assuming class 1 = bot.
class_labels = best_model.named_steps["logreg"].classes_
class_1_index = list(class_labels).index(1)
y_proba_class_1 = y_proba[:, class_1_index]

try:
    test_roc_auc = roc_auc_score(y_test_encoded, y_proba_class_1)
except Exception as e:
    test_roc_auc = None
    print("Could not compute ROC-AUC:", e)


# =========================================================
# Save results as JSON
# =========================================================

results = {
    "model_type": "TF-IDF + LogisticRegression",
    "best_cv_f1_score": search.best_score_,
    "test_roc_auc": test_roc_auc,
    "best_parameters": search.best_params_,
    "classification_report": report_dict,
    "confusion_matrix": conf_matrix
}

results_filename = "logreg_search_results.json"

with open(results_filename, "w") as f:
    json.dump(
        results,
        f,
        indent=4,
        default=make_json_serializable
    )

print(f"Saved results to {results_filename}")


# =========================================================
# Save predicted probabilities
# =========================================================

probability_df = pd.DataFrame({
    "true_label": y_test_encoded,
    "predicted_label": y_pred,
    "probability_class_0": y_proba[:, 0],
    "probability_class_1": y_proba[:, 1]
})

probability_filename = "logreg_test_predicted_probabilities.csv"
probability_df.to_csv(probability_filename, index=False)

print(f"Saved predicted probabilities to {probability_filename}")


# =========================================================
# Print summary
# =========================================================

print("\nBest CV F1 score:")
print(search.best_score_)

print("\nBest parameters:")
print(search.best_params_)

print("\nTest set classification report:")
print(report_text)

print("\nConfusion matrix:")
print(conf_matrix)

print("\nTest ROC-AUC:")
print(test_roc_auc)


# =========================================================
# Save best model
# =========================================================

model_filename = "tfidf_logreg_bot_model.joblib"
joblib.dump(best_model, model_filename, compress=3)

print(f"Saved best model to {model_filename}")


# =========================================================
# Clean up memory
# =========================================================

gc.collect()

Starting Logistic Regression RandomizedSearchCV...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 2/5] END features__char_tfidf__max_df=0.9, features__char_tfidf__max_features=80000, features__char_tfidf__min_df=5, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 1.0, 'char_tfidf': 1.25}, features__word_tfidf__max_df=0.9, features__word_tfidf__max_features=80000, features__word_tfidf__min_df=3, features__word_tfidf__ngram_range=(1, 1), features__word_tfidf__sublinear_tf=True, logreg__C=0.0031613645505296157, logreg__class_weight=None, logreg__l1_ratio=0.9, logreg__max_iter=3000, logreg__penalty=elasticnet, logreg__tol=0.0001;, score=0.667 total time=   3.0s
[CV 4/5] END features__char_tfidf__max_df=0.9, features__char_tfidf__max_features=80000, features__char_tfidf__min_df=5, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'

/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV 3/5] END features__char_tfidf__max_df=1.0, features__char_tfidf__max_features=80000, features__char_tfidf__min_df=3, features__char_tfidf__ngram_range=(3, 5), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 0.75, 'char_tfidf': 1.0}, features__word_tfidf__max_df=0.9, features__word_tfidf__max_features=80000, features__word_tfidf__min_df=5, features__word_tfidf__ngram_range=(1, 2), features__word_tfidf__sublinear_tf=True, logreg__C=0.0013399717231179719, logreg__class_weight=None, logreg__max_iter=3000, logreg__penalty=l1, logreg__tol=0.001;, score=0.000 total time=   3.2s
[CV 2/5] END features__char_tfidf__max_df=1.0, features__char_tfidf__max_features=20000, features__char_tfidf__min_df=2, features__char_tfidf__ngram_range=(3, 5), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 1.0, 'char_tfidf': 1.0}, features__word_tfidf__max_df=1.0, features__word_tfidf__max_features=20000, features__word_tfidf__min_df=3

/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Search complete.
Saved results to logreg_search_results.json
Saved predicted probabilities to logreg_test_predicted_probabilities.csv

Best CV F1 score:
0.8443569600532316

Best parameters:
{'features__char_tfidf__max_df': 1.0, 'features__char_tfidf__max_features': 80000, 'features__char_tfidf__min_df': 3, 'features__char_tfidf__ngram_range': (3, 5), 'features__char_tfidf__sublinear_tf': True, 'features__transformer_weights': {'word_tfidf': 1.0, 'char_tfidf': 1.25}, 'features__word_tfidf__max_df': 0.9, 'features__word_tfidf__max_features': 80000, 'features__word_tfidf__min_df': 2, 'features__word_tfidf__ngram_range': (1, 2), 'features__word_tfidf__sublinear_tf': True, 'logreg__C': 2.304582609673582, 'logreg__class_weight': 'balanced', 'logreg__max_iter': 1000, 'logreg__penalty': 'l1', 'logreg__tol': 0.001}

Test set classification report:
              precision    recall  f1-score   support

           0       0.86      0.81      0.83      1278
           1       0.82      0.87      0

0

[CV 1/5] END features__char_tfidf__max_df=0.9, features__char_tfidf__max_features=50000, features__char_tfidf__min_df=3, features__char_tfidf__ngram_range=(4, 6), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 1.0, 'char_tfidf': 0.5}, features__word_tfidf__max_df=0.95, features__word_tfidf__max_features=20000, features__word_tfidf__min_df=5, features__word_tfidf__ngram_range=(1, 2), features__word_tfidf__sublinear_tf=True, logreg__C=7.510418138777536, logreg__class_weight=balanced, logreg__max_iter=2000, logreg__penalty=l1, logreg__tol=0.0001;, score=0.785 total time= 6.8min
[CV 5/5] END features__char_tfidf__max_df=0.9, features__char_tfidf__max_features=20000, features__char_tfidf__min_df=3, features__char_tfidf__ngram_range=(3, 5), features__char_tfidf__sublinear_tf=True, features__transformer_weights={'word_tfidf': 1.0, 'char_tfidf': 1.25}, features__word_tfidf__max_df=1.0, features__word_tfidf__max_features=80000, features__word_tfidf__min_df